# Module 03 — Unsupervised & Statistical Learning
## 03-07: Anomaly Detection

**Objective:** Implement multiple anomaly detection methods from scratch — statistical rules (z-score, IQR), multivariate Gaussian density estimation, and Isolation Forest — and evaluate them with precision-recall curves appropriate for imbalanced anomaly problems.

**Prerequisites:** 1-07 (Probability & Statistics)

## Part 0 — Setup & Prerequisites

This notebook covers anomaly detection (also called outlier detection or novelty detection). We build a progression of methods from simple univariate rules to a tree-based ensemble, culminating in an evaluation harness that correctly handles the heavily imbalanced nature of anomaly data.

**Prerequisites:** 1-07 (Probability & Statistics)

In [ ]:
import sys
import warnings
import random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import multivariate_normal, norm as scipy_norm
from scipy.special import logsumexp as lse
from sklearn.datasets import make_classification
from sklearn.ensemble import IsolationForest as SklearnIF
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    f1_score,
    confusion_matrix,
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch

print(f'Python : {sys.version.split()[0]}')
print(f'NumPy  : {np.__version__}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Experiment constants ──────────────────────────────────────────────────────
N_SAMPLES       = 1000
N_FEATURES      = 2        # 2-D for visualisation
ANOMALY_FRAC    = 0.05     # 5% anomalies
N_TREES         = 100      # Isolation Forest ensemble size
MAX_SAMPLES_IF  = 256      # sub-sample size per tree
N_THRESHOLDS    = 200      # precision-recall sweep resolution
REG_COV         = 1e-6     # covariance regularisation

# ── Plot style ────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
COL_NORMAL  = 'steelblue'
COL_ANOMALY = 'tomato'
print('Configuration loaded.')

### Data Loading & EDA

We use `make_classification` to generate a labelled dataset where class 1 represents anomalies (5% of data). In real applications labels are not available — we use them only for evaluation. The data is standardised so that statistical thresholds have interpretable units (standard deviations).

In [ ]:
# ── Generate dataset ──────────────────────────────────────────────────────────
n_anomalies = int(N_SAMPLES * ANOMALY_FRAC)
n_normals   = N_SAMPLES - n_anomalies

X_raw, y = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=N_FEATURES,
    n_redundant=0,
    n_clusters_per_class=1,
    weights=[1 - ANOMALY_FRAC, ANOMALY_FRAC],
    flip_y=0.0,
    class_sep=1.5,
    random_state=SEED,
)

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f'Dataset shape   : {X.shape}')
print(f'Normal points   : {(y == 0).sum()} ({(y==0).mean()*100:.1f}%)')
print(f'Anomaly points  : {(y == 1).sum()} ({(y==1).mean()*100:.1f}%)')
print(f'Feature range   : [{X.min():.3f}, {X.max():.3f}]')

# ── Train / test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)
print(f'\nTrain: {X_train.shape[0]}  (anomalies: {y_train.sum()})')
print(f'Test : {X_test.shape[0]}   (anomalies: {y_test.sum()})')

# ── Scatter EDA ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.scatter(X[y == 0, 0], X[y == 0, 1], c=COL_NORMAL,  alpha=0.4, s=20,
           label='Normal')
ax.scatter(X[y == 1, 0], X[y == 1, 1], c=COL_ANOMALY, alpha=0.8, s=50,
           marker='x', lw=1.5, label='Anomaly')
ax.set_title('Ground Truth Labels', fontsize=12)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()

ax = axes[1]
ax.scatter(X[:, 0], X[:, 1], c='slategray', alpha=0.3, s=15)
ax.set_title('Unlabelled Data (Deployed Setting)', fontsize=12)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')

fig.suptitle('Anomaly Detection Dataset (5% Anomalies)', fontsize=14,
             fontweight='bold')
plt.tight_layout()
plt.show()

# ── Feature distributions ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, N_FEATURES, figsize=(10, 3))
for i, ax in enumerate(axes):
    ax.hist(X[y == 0, i], bins=25, alpha=0.5, color=COL_NORMAL,
            density=True, label='Normal')
    ax.hist(X[y == 1, i], bins=12, alpha=0.7, color=COL_ANOMALY,
            density=True, label='Anomaly')
    ax.set_title(f'Feature {i + 1} distribution', fontsize=11)
    ax.set_xlabel('Standardised value')
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## Part 1 — Statistical Anomaly Detectors from Scratch

### 1.1 Univariate Rules: Z-Score and IQR

The simplest approach treats each feature independently:

**Z-score rule:** a point is anomalous if any feature exceeds $|z| > \tau$ standard deviations:
$$z_i = \frac{x_i - \mu_i}{\sigma_i}, \quad \text{anomaly if } \max_i |z_i| > \tau$$

**IQR rule:** uses quartiles to define a robust outlier band (unaffected by the outliers themselves):
$$\text{anomaly if } x_i < Q_1 - 1.5 \cdot \text{IQR} \; \text{ or } \; x_i > Q_3 + 1.5 \cdot \text{IQR}$$

### 1.2 Multivariate Gaussian Density

For correlated features, use the full multivariate Gaussian log-density as the anomaly score. A point is anomalous if its log-density is below a threshold $\epsilon$:
$$\log p(\mathbf{x}) = -\frac{d}{2}\log(2\pi) - \frac{1}{2}\log|\boldsymbol{\Sigma}| - \frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1}(\mathbf{x}-\boldsymbol{\mu})$$

The Mahalanobis distance $D^2 = (\mathbf{x}-\boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1}(\mathbf{x}-\boldsymbol{\mu})$ provides an equivalent scalar score.

### 1.3 Isolation Forest Intuition

Isolation Forest exploits the observation that anomalies are few and different — they are isolated by random splits in far fewer steps than normal points. The **anomaly score** is the inverse average path length across an ensemble of isolation trees.

In [ ]:
def zscore_anomaly_scores(
    X_train: np.ndarray,
    X_test: np.ndarray,
) -> np.ndarray:
    '''Compute per-sample anomaly scores using the max absolute z-score rule.

    Fits mean and std on X_train (normal data only in deployment), then
    scores each test point by its maximum standardised feature value.

    Args:
        X_train: Training data of shape (n_train, d).
        X_test:  Test data of shape (n_test, d).

    Returns:
        Anomaly scores of shape (n_test,). Higher = more anomalous.
    '''
    mu    = X_train.mean(axis=0)          # (d,)
    sigma = X_train.std(axis=0) + 1e-10   # (d,) avoid div-by-zero
    z     = (X_test - mu) / sigma         # (n_test, d)
    return np.max(np.abs(z), axis=1)      # max absolute z across features


def iqr_anomaly_scores(
    X_train: np.ndarray,
    X_test: np.ndarray,
    iqr_factor: float = 1.5,
) -> np.ndarray:
    '''Compute per-sample anomaly scores using the IQR fence rule.

    Score = maximum exceedance above upper fence or below lower fence,
    across all features.  Score <= 0 means no fence is violated.

    Args:
        X_train:    Training data of shape (n_train, d).
        X_test:     Test data of shape (n_test, d).
        iqr_factor: Multiplier for IQR (default 1.5 = Tukey's rule).

    Returns:
        Anomaly scores of shape (n_test,). Higher = more anomalous.
    '''
    Q1  = np.percentile(X_train, 25, axis=0)   # (d,)
    Q3  = np.percentile(X_train, 75, axis=0)   # (d,)
    IQR = Q3 - Q1                              # (d,)
    lower = Q1 - iqr_factor * IQR
    upper = Q3 + iqr_factor * IQR
    # Exceedance above upper fence or below lower fence per feature
    above = np.maximum(X_test - upper, 0.0)    # (n_test, d)
    below = np.maximum(lower - X_test, 0.0)    # (n_test, d)
    return np.max(above + below, axis=1)       # worst-case feature violation


# ── Compute scores on test set ────────────────────────────────────────────────
scores_z   = zscore_anomaly_scores(X_train, X_test)
scores_iqr = iqr_anomaly_scores(X_train, X_test)

print('Z-score scores on test set:')
print(f'  Normal  — mean={scores_z[y_test==0].mean():.3f},  '
      f'max={scores_z[y_test==0].max():.3f}')
print(f'  Anomaly — mean={scores_z[y_test==1].mean():.3f},  '
      f'max={scores_z[y_test==1].max():.3f}')
print('IQR scores on test set:')
print(f'  Normal  — mean={scores_iqr[y_test==0].mean():.3f},  '
      f'max={scores_iqr[y_test==0].max():.3f}')
print(f'  Anomaly — mean={scores_iqr[y_test==1].mean():.3f},  '
      f'max={scores_iqr[y_test==1].max():.3f}')

### Multivariate Gaussian Density Estimator

The univariate rules treat features independently and miss correlations. The multivariate Gaussian fits a single ellipsoidal density to the normal training data. Any point far from the fitted distribution (high Mahalanobis distance) is flagged as anomalous.

We implement the Mahalanobis distance scorer using the Cholesky decomposition from 03-06, which provides a numerically stable log-determinant.

In [ ]:
class GaussianAnomalyDetector:
    '''Anomaly detector based on multivariate Gaussian density estimation.

    Fits a single Gaussian to (presumed) normal training data. At test
    time, assigns anomaly scores equal to the Mahalanobis distance from
    the fitted distribution — equivalently the negative log-density.

    Attributes:
        mu_:    Fitted mean vector, shape (d,).
        Sigma_: Fitted covariance matrix, shape (d, d).
        L_:     Cholesky factor of regularised covariance, shape (d, d).
    '''

    def __init__(self, reg: float = REG_COV) -> None:
        '''Initialise with covariance regularisation.

        Args:
            reg: Diagonal regularisation to ensure positive definiteness.
        '''
        self.reg    = reg
        self.mu_    = None
        self.Sigma_ = None
        self.L_     = None

    def fit(self, X: np.ndarray) -> 'GaussianAnomalyDetector':
        '''Estimate mean and covariance from training data.

        Args:
            X: Training data of shape (n, d). Should contain normal points only.

        Returns:
            self — fitted estimator.
        '''
        self.mu_    = X.mean(axis=0)
        self.Sigma_ = np.cov(X.T) + self.reg * np.eye(X.shape[1])
        self.L_     = np.linalg.cholesky(self.Sigma_)
        return self

    def mahalanobis(self, X: np.ndarray) -> np.ndarray:
        '''Compute squared Mahalanobis distances from the fitted mean.

        D^2 = (x - mu)^T Sigma^{-1} (x - mu)

        Uses the Cholesky factor for efficient, stable computation.

        Args:
            X: Data of shape (n, d).

        Returns:
            Squared Mahalanobis distances of shape (n,).
        '''
        diff = X - self.mu_                       # (n, d)
        z    = np.linalg.solve(self.L_, diff.T)   # (d, n)
        return np.sum(z ** 2, axis=0)             # (n,)

    def log_density(self, X: np.ndarray) -> np.ndarray:
        '''Compute log-density of the fitted Gaussian at each point.

        Args:
            X: Data of shape (n, d).

        Returns:
            Log-densities of shape (n,). More negative = more anomalous.
        '''
        d        = X.shape[1]
        log_det  = 2.0 * np.sum(np.log(np.diag(self.L_)))
        maha_sq  = self.mahalanobis(X)
        return -0.5 * (d * np.log(2.0 * np.pi) + log_det + maha_sq)

    def score_samples(self, X: np.ndarray) -> np.ndarray:
        '''Compute anomaly scores (higher = more anomalous).

        Score is the squared Mahalanobis distance; equivalent to using
        negative log-density as the anomaly score.

        Args:
            X: Data of shape (n, d).

        Returns:
            Anomaly scores of shape (n,).
        '''
        return self.mahalanobis(X)

    def predict(self, X: np.ndarray, threshold: float = 5.0) -> np.ndarray:
        '''Classify points as normal (0) or anomaly (1).

        Args:
            X:         Data of shape (n, d).
            threshold: Mahalanobis distance threshold; default 5.0 corresponds
                       roughly to 3-sigma for a 2-D Gaussian.

        Returns:
            Binary predictions of shape (n,).
        '''
        return (self.score_samples(X) > threshold).astype(int)


# ── Fit on training normals only and score test set ───────────────────────────
gauss_det = GaussianAnomalyDetector(reg=REG_COV)
gauss_det.fit(X_train[y_train == 0])   # fit on normals only
scores_gauss = gauss_det.score_samples(X_test)

print('Gaussian Mahalanobis scores on test set:')
print(f'  Normal  — mean={scores_gauss[y_test==0].mean():.3f},  '
      f'p95={np.percentile(scores_gauss[y_test==0], 95):.3f}')
print(f'  Anomaly — mean={scores_gauss[y_test==1].mean():.3f},  '
      f'p5={np.percentile(scores_gauss[y_test==1], 5):.3f}')

### Isolation Forest from Scratch

Isolation Forest builds an ensemble of **isolation trees** where each tree recursively partitions the feature space with random axis-aligned splits. The intuition:

- **Normal points** lie in dense regions — they require many splits to isolate
- **Anomalies** lie far from clusters — they are isolated in very few splits

The anomaly score for point $\mathbf{x}$ is:
$$s(\mathbf{x}, n) = 2^{-\frac{\overline{h}(\mathbf{x})}{c(n)}}$$

where $\overline{h}(\mathbf{x})$ is the mean path length across trees and $c(n)$ is the expected path length of an unsuccessful BST search on $n$ points:
$$c(n) = 2H(n-1) - \frac{2(n-1)}{n}, \quad H(i) \approx \ln(i) + 0.5772$$

In [ ]:
def _c_factor(n: int) -> float:
    '''Expected path length of an unsuccessful BST search for n samples.

    Normalisation constant used to convert raw path lengths to anomaly
    scores in the range (0, 1). From Liu et al. (2008) Equation 1.

    Args:
        n: Number of samples in the isolation tree sub-sample.

    Returns:
        Expected path length c(n) as a float.
    '''
    if n <= 1:
        return 0.0
    euler_mascheroni = 0.5772156649
    H_n_minus_1 = np.log(n - 1) + euler_mascheroni   # harmonic number approx
    return 2.0 * H_n_minus_1 - 2.0 * (n - 1) / n


class IsolationTree:
    '''A single isolation tree node built by recursive random partitioning.

    Splits the current sub-sample on a random feature at a random split
    point chosen uniformly within the feature range. Recursion stops at
    max_depth or when fewer than 2 samples remain.

    Attributes:
        is_leaf:      Whether this is a terminal node.
        size:         Number of samples at this node.
        split_feature: Feature index used for splitting (None if leaf).
        split_value:  Split threshold (None if leaf).
        left:         Left child (x <= split_value).
        right:        Right child (x > split_value).
    '''

    def __init__(
        self,
        X: np.ndarray,
        depth: int = 0,
        max_depth: int = None,
        rng: np.random.RandomState = None,
    ) -> None:
        '''Recursively build an isolation tree on the given sub-sample.

        Args:
            X:         Data sub-sample of shape (n, d).
            depth:     Current tree depth (root = 0).
            max_depth: Maximum depth before forcing a leaf.
            rng:       RandomState for reproducible splits.
        '''
        self.size      = X.shape[0]
        self.is_leaf   = False
        self.split_feature = None
        self.split_value   = None
        self.left          = None
        self.right         = None

        if rng is None:
            rng = np.random.RandomState(SEED)

        # Stop at max depth or when isolation is complete
        if max_depth is not None and depth >= max_depth:
            self.is_leaf = True
            return
        if X.shape[0] <= 1:
            self.is_leaf = True
            return

        # Choose a random feature and a random split within its range
        d = X.shape[1]
        q = int(rng.randint(0, d))
        feat_min = X[:, q].min()
        feat_max = X[:, q].max()
        if feat_min == feat_max:
            self.is_leaf = True
            return

        p = rng.uniform(feat_min, feat_max)
        self.split_feature = q
        self.split_value   = p

        left_mask  = X[:, q] <= p
        right_mask = ~left_mask
        self.left  = IsolationTree(X[left_mask],  depth + 1, max_depth, rng)
        self.right = IsolationTree(X[right_mask], depth + 1, max_depth, rng)

    def path_length(self, x: np.ndarray, depth: int = 0) -> float:
        '''Compute the path length to isolate a single point.

        If the point reaches a leaf with more than one sample remaining,
        the path length is augmented by c(leaf_size) to account for
        the expected additional splits needed.

        Args:
            x:     Single data point of shape (d,).
            depth: Accumulated depth (used internally in recursion).

        Returns:
            Path length (float).
        '''
        if self.is_leaf or self.split_feature is None:
            return depth + _c_factor(self.size)
        if x[self.split_feature] <= self.split_value:
            return self.left.path_length(x, depth + 1)
        return self.right.path_length(x, depth + 1)

In [ ]:
class IsolationForest:
    '''Isolation Forest anomaly detector (Liu et al., 2008).

    Builds an ensemble of isolation trees on sub-sampled data.
    Anomaly scores in (0, 1): scores near 1 indicate anomalies;
    scores near 0.5 indicate normal points.

    Attributes:
        trees_:       List of fitted IsolationTree objects.
        n_samples_:   Sub-sample size used to build each tree.
        max_depth_:   Maximum depth of each tree.
        c_factor_:    Normalisation constant c(n_samples_).
    '''

    def __init__(
        self,
        n_estimators: int = N_TREES,
        max_samples: int = MAX_SAMPLES_IF,
        random_state: int = SEED,
    ) -> None:
        '''Initialise Isolation Forest hyperparameters.

        Args:
            n_estimators: Number of isolation trees.
            max_samples:  Sub-sample size per tree.
            random_state: Random seed for reproducibility.
        '''
        self.n_estimators = n_estimators
        self.max_samples  = max_samples
        self.random_state = random_state
        self.trees_       = []
        self.n_samples_   = None
        self.max_depth_   = None
        self.c_factor_    = None

    def fit(self, X: np.ndarray) -> 'IsolationForest':
        '''Build the isolation forest on training data.

        Each tree is grown on a random sub-sample of size min(max_samples, n).
        Maximum tree depth is set to ceil(log2(n_samples)) to limit path
        length computations.

        Args:
            X: Training data of shape (n, d).

        Returns:
            self — fitted estimator.
        '''
        rng                = np.random.RandomState(self.random_state)
        n                  = X.shape[0]
        self.n_samples_    = min(self.max_samples, n)
        self.max_depth_    = int(np.ceil(np.log2(self.n_samples_)))
        self.c_factor_     = _c_factor(self.n_samples_)
        self.trees_        = []

        for t in range(self.n_estimators):
            sub_idx   = rng.choice(n, size=self.n_samples_, replace=False)
            X_sub     = X[sub_idx]
            tree_rng  = np.random.RandomState(rng.randint(0, 100_000))
            tree      = IsolationTree(
                X_sub, depth=0, max_depth=self.max_depth_, rng=tree_rng
            )
            self.trees_.append(tree)

        return self

    def _mean_path_lengths(self, X: np.ndarray) -> np.ndarray:
        '''Compute the mean path length across all trees for each point.

        Args:
            X: Data of shape (n, d).

        Returns:
            Mean path lengths of shape (n,).
        '''
        n = X.shape[0]
        path_lengths = np.zeros((n, self.n_estimators))
        for t, tree in enumerate(self.trees_):
            for i in range(n):
                path_lengths[i, t] = tree.path_length(X[i], depth=0)
        return path_lengths.mean(axis=1)

    def score_samples(self, X: np.ndarray) -> np.ndarray:
        '''Compute anomaly scores in range (0, 1).

        Score = 2^{-mean_path_length / c(n_samples_)}
        Scores near 1.0 indicate anomalies; scores near 0.5 are normal.

        Args:
            X: Data of shape (n, d).

        Returns:
            Anomaly scores of shape (n,).
        '''
        mean_pl = self._mean_path_lengths(X)
        if self.c_factor_ == 0.0:
            return np.ones(X.shape[0]) * 0.5
        return 2.0 ** (-mean_pl / self.c_factor_)

    def predict(self, X: np.ndarray, threshold: float = 0.55) -> np.ndarray:
        '''Classify each point as normal (0) or anomaly (1).

        Args:
            X:         Data of shape (n, d).
            threshold: Score threshold above which a point is anomalous.

        Returns:
            Binary predictions of shape (n,).
        '''
        return (self.score_samples(X) >= threshold).astype(int)


# ── Fit our Isolation Forest on all training data (unsupervised) ──────────────
print('Fitting Isolation Forest (this takes ~30s for 100 trees)...')
if_scratch = IsolationForest(n_estimators=N_TREES, max_samples=MAX_SAMPLES_IF,
                             random_state=SEED)
if_scratch.fit(X_train)
scores_if = if_scratch.score_samples(X_test)
print(f'Done.  Score range: [{scores_if.min():.4f}, {scores_if.max():.4f}]')
print(f'  Normal  — mean score={scores_if[y_test==0].mean():.4f}')
print(f'  Anomaly — mean score={scores_if[y_test==1].mean():.4f}')

### 1.4 Local Outlier Factor (LOF)

Unlike global methods (Gaussian, Isolation Forest), **LOF** measures anomalousness relative to the *local neighbourhood density*. A point is anomalous if its local density is much lower than its neighbours'.

For each point $\mathbf{x}_i$, the **local reachability density** is:
$$\text{lrd}_k(i) = \left(\frac{1}{k}\sum_{j \in N_k(i)} \text{reach-dist}_k(i, j)\right)^{-1}$$

where $\text{reach-dist}_k(i, j) = \max(k\text{-dist}(j),\ d(i, j))$ smooths distances to avoid numerical issues for very close neighbours.

The **LOF score** for point $i$ compares its lrd against its neighbours:
$$\text{LOF}_k(i) = \frac{\sum_{j \in N_k(i)} \text{lrd}_k(j)}{k \cdot \text{lrd}_k(i)}$$

- $\text{LOF} \approx 1$: point has similar density to neighbours (normal)
- $\text{LOF} \gg 1$: point is in a sparser region than its neighbours (anomaly)

In [ ]:
def compute_knn_distances_and_indices(
    X: np.ndarray,
    k: int,
) -> tuple:
    '''Compute k-nearest neighbour distances and indices for all points.

    Uses a vectorised pairwise distance matrix. For large n, consider
    approximate methods (e.g. sklearn BallTree).

    Args:
        X: Data of shape (n, d).
        k: Number of nearest neighbours.

    Returns:
        knn_dists:  Sorted nearest-neighbour distances, shape (n, k).
        knn_idx:    Corresponding indices into X, shape (n, k).
    '''
    n = X.shape[0]
    # Vectorised squared pairwise distances: (n, n)
    diff_sq    = np.sum((X[:, None] - X[None]) ** 2, axis=2)
    np.fill_diagonal(diff_sq, np.inf)   # exclude self
    knn_idx    = np.argsort(diff_sq, axis=1)[:, :k]        # (n, k)
    knn_dists  = np.sqrt(np.take_along_axis(diff_sq, knn_idx, axis=1))  # (n, k)
    return knn_dists, knn_idx


class LocalOutlierFactor:
    '''Local Outlier Factor anomaly detector (Breunig et al., 2000).

    Measures anomalousness relative to the local neighbourhood density.
    A LOF score >> 1 indicates that a point is in a much sparser region
    than its k-nearest neighbours — a hallmark of anomalies.

    Attributes:
        n_neighbors_:  Number of neighbours used.
        X_train_:      Copy of training data, shape (n, d).
        knn_dists_:    k-NN distances for each training point, shape (n, k).
        knn_idx_:      k-NN indices for each training point, shape (n, k).
        lrd_:          Local reachability densities, shape (n,).
        k_dist_:       k-th nearest neighbour distances, shape (n,).
    '''

    def __init__(self, n_neighbors: int = 20) -> None:
        '''Initialise LOF with k-nearest neighbours.

        Args:
            n_neighbors: Number of neighbours k for density estimation.
        '''
        self.n_neighbors  = n_neighbors
        self.X_train_     = None
        self.knn_dists_   = None
        self.knn_idx_     = None
        self.lrd_         = None
        self.k_dist_      = None

    def fit(self, X: np.ndarray) -> 'LocalOutlierFactor':
        '''Precompute k-NN and local reachability densities for training data.

        Args:
            X: Training data of shape (n, d).

        Returns:
            self — fitted estimator.
        '''
        self.X_train_   = X.copy()
        self.knn_dists_, self.knn_idx_ = compute_knn_distances_and_indices(
            X, self.n_neighbors
        )
        self.k_dist_ = self.knn_dists_[:, -1]    # (n,) k-th distance = k-dist
        n            = X.shape[0]
        k            = self.n_neighbors
        self.lrd_    = np.zeros(n)

        for i in range(n):
            neighbors_idx  = self.knn_idx_[i]          # indices of k neighbours
            # reach-dist_k(i, j) = max(k-dist(j), d(i, j))
            reach_dists    = np.maximum(
                self.k_dist_[neighbors_idx],            # k-dist(j)
                self.knn_dists_[i],                     # d(i, j)
            )
            self.lrd_[i] = 1.0 / (np.mean(reach_dists) + 1e-10)

        return self

    def score_samples(self, X: np.ndarray) -> np.ndarray:
        '''Compute LOF scores for test points (higher = more anomalous).

        For each test point, finds its k nearest training neighbours,
        computes its local reachability density, then compares it to
        the densities of its neighbours.

        Args:
            X: Test data of shape (n_test, d).

        Returns:
            LOF scores of shape (n_test,).
        '''
        n_test = X.shape[0]
        k      = self.n_neighbors
        scores = np.zeros(n_test)

        for i in range(n_test):
            # Distances from this test point to all training points
            dists      = np.linalg.norm(self.X_train_ - X[i], axis=1)
            sorted_idx = np.argsort(dists)[:k]
            knn_d      = dists[sorted_idx]

            # Reach distances and lrd of this test point
            reach_dists = np.maximum(self.k_dist_[sorted_idx], knn_d)
            lrd_test    = 1.0 / (np.mean(reach_dists) + 1e-10)

            # LOF = mean lrd of neighbours / lrd of test point
            lrd_neighbours = self.lrd_[sorted_idx]
            scores[i]      = np.mean(lrd_neighbours) / (lrd_test + 1e-10)

        return scores

    def predict(self, X: np.ndarray, threshold: float = 1.5) -> np.ndarray:
        '''Classify points as normal (0) or anomaly (1).

        Args:
            X:         Data of shape (n, d).
            threshold: LOF score above which a point is anomalous.

        Returns:
            Binary predictions of shape (n,).
        '''
        return (self.score_samples(X) >= threshold).astype(int)


# ── Fit LOF on normal training points and score test set ──────────────────────
print('Fitting LOF (n_neighbors=20)...')
lof_detector = LocalOutlierFactor(n_neighbors=20)
lof_detector.fit(X_train[y_train == 0])
scores_lof = lof_detector.score_samples(X_test)
print(f'Done.  Score range: [{scores_lof.min():.4f}, {scores_lof.max():.4f}]')
print(f'  Normal  — mean LOF={scores_lof[y_test==0].mean():.3f}')
print(f'  Anomaly — mean LOF={scores_lof[y_test==1].mean():.3f}')

result_lof = evaluate_anomaly_detector(scores_lof, y_test, 'LOF (scratch)')
all_results.append(result_lof)

# ── LOF training-data distribution (sanity check: should be near 1.0) ────────
lof_train_scores = lof_detector.lrd_
print(f'\nTraining lrd stats (should be positive and well-conditioned):')
print(f'  min={lof_train_scores.min():.4f}, max={lof_train_scores.max():.4f}, '
      f'mean={lof_train_scores.mean():.4f}')

### 1.5 Kernel Density Estimation (KDE) Anomaly Detector

KDE estimates the data density non-parametrically by placing a Gaussian kernel at each training point:
$$\hat{p}(\\mathbf{x}) = \\frac{1}{n h^d} \\sum_{i=1}^{n} K\\!\\left(\\frac{\\|\\mathbf{x} - \\mathbf{x}_i\\|}{h}\\right), \\quad K(u) = \\frac{1}{(2\\pi)^{d/2}} e^{-u^2/2}$$

The bandwidth $h$ controls the smoothness: small $h$ is spiky (overfits); large $h$ over-smooths. Silverman's rule provides a data-driven default: $h = \\sigma \\cdot n^{-1/(d+4)}$.

In [ ]:
class GaussianKDEAnomalyDetector:
    '''Anomaly detector based on Gaussian kernel density estimation.

    Estimates the density of normal training data non-parametrically.
    Points with low estimated density are flagged as anomalies.
    Bandwidth is set via Silverman's rule or a user-supplied value.

    Attributes:
        X_train_:   Training data used as kernel centres, shape (n, d).
        bandwidth_: Kernel bandwidth h (scalar).
        log_norm_:  Log-normalisation constant for each kernel.
    '''

    def __init__(self, bandwidth: float | None = None) -> None:
        '''Initialise KDE detector with optional fixed bandwidth.

        Args:
            bandwidth: Kernel bandwidth. If None, Silverman's rule is used.
        '''
        self.bandwidth  = bandwidth
        self.X_train_   = None
        self.bandwidth_ = None
        self.log_norm_  = None

    def _silverman_bandwidth(self, X: np.ndarray) -> float:
        '''Compute Silverman rule-of-thumb bandwidth: h = sigma * n^{-1/(d+4)}.

        Args:
            X: Training data of shape (n, d).

        Returns:
            Bandwidth scalar h.
        '''
        n, d  = X.shape
        sigma = float(np.mean(X.std(axis=0)))
        return sigma * (n ** (-1.0 / (d + 4.0)))

    def fit(self, X: np.ndarray) -> 'GaussianKDEAnomalyDetector':
        '''Store training points as kernel centres and compute bandwidth.

        Args:
            X: Training data of shape (n, d).

        Returns:
            self — fitted estimator.
        '''
        self.X_train_   = X.copy()
        if self.bandwidth is None:
            self.bandwidth_ = self._silverman_bandwidth(X)
        else:
            self.bandwidth_ = float(self.bandwidth)
        d = X.shape[1]
        self.log_norm_ = -d * np.log(self.bandwidth_) - 0.5 * d * np.log(2.0 * np.pi)
        return self

    def log_density(self, X: np.ndarray) -> np.ndarray:
        '''Evaluate KDE log-density via log-sum-exp for numerical stability.

        Args:
            X: Test data of shape (n_test, d).

        Returns:
            Log-density values of shape (n_test,).
        '''
        h     = self.bandwidth_
        diffs = X[:, None] - self.X_train_[None]         # (n_test, n_train, d)
        sq_d  = np.sum(diffs ** 2, axis=2) / (h ** 2)    # (n_test, n_train)
        log_k = -0.5 * sq_d                              # unnormalised log-kernel
        log_dens = lse(log_k, axis=1) + self.log_norm_ - np.log(self.X_train_.shape[0])
        return log_dens

    def score_samples(self, X: np.ndarray) -> np.ndarray:
        '''Anomaly score = negative log-density (higher = more anomalous).

        Args:
            X: Data of shape (n, d).

        Returns:
            Anomaly scores of shape (n,).
        '''
        return -self.log_density(X)

    def predict(self, X: np.ndarray, threshold: float | None = None) -> np.ndarray:
        '''Classify points as normal (0) or anomaly (1).

        Args:
            X:         Data of shape (n, d).
            threshold: Score threshold. If None, uses the 95th percentile
                       of scores on the training data.

        Returns:
            Binary predictions of shape (n,).
        '''
        train_scores = self.score_samples(self.X_train_)
        t = threshold if threshold is not None else float(np.percentile(train_scores, 95))
        return (self.score_samples(X) >= t).astype(int)


# ── Fit KDE on normal training points ─────────────────────────────────────────
kde_det    = GaussianKDEAnomalyDetector()
kde_det.fit(X_train[y_train == 0])
scores_kde = kde_det.score_samples(X_test)
kde_bw     = kde_det.bandwidth_
print(f'KDE bandwidth (Silverman): {kde_bw:.4f}')
print(f'  Normal  — mean neg-log-dens={scores_kde[y_test==0].mean():.3f}')
print(f'  Anomaly — mean neg-log-dens={scores_kde[y_test==1].mean():.3f}')

result_kde = evaluate_anomaly_detector(scores_kde, y_test, 'KDE (Gaussian)')
all_results.append(result_kde)

# ── Bandwidth sensitivity ─────────────────────────────────────────────────────
bandwidths  = [0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
kde_bw_rows = []
for bw in bandwidths:
    kde_bw_inst = GaussianKDEAnomalyDetector(bandwidth=bw)
    kde_bw_inst.fit(X_train[y_train == 0])
    s_bw   = kde_bw_inst.score_samples(X_test)
    ap_bw  = float(average_precision_score(y_test, s_bw))
    kde_bw_rows.append({'bandwidth': bw, 'AP': round(ap_bw, 4)})
    print(f'  bandwidth={bw:.2f}: AP={ap_bw:.4f}')
kde_silverman_ap = result_kde['ap']
print(f'  Silverman ({kde_bw:.3f}): AP={kde_silverman_ap:.4f}')

fig, ax = plt.subplots(figsize=(7, 3.5))
bws  = [r['bandwidth'] for r in kde_bw_rows]
aps  = [r['AP'] for r in kde_bw_rows]
ax.semilogx(bws, aps, 'o-', color='steelblue', lw=2)
ax.axhline(kde_silverman_ap, color='tomato', linestyle='--', lw=1.5,
           label=f'Silverman h={kde_det.bandwidth_:.3f}')
ax.set_xlabel('Bandwidth h')
ax.set_ylabel('Average Precision')
ax.set_title('KDE Bandwidth Sensitivity', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## Part 2 — Putting It All Together

We now assemble a unified evaluation harness. Anomaly detection evaluation differs from standard classification:
- **Precision-Recall** curves are preferred over ROC because the positive class (anomaly) is rare
- **Average Precision (AP)** summarises the PR curve as a single number
- We tune each method's decision threshold separately to find the best F1

In [ ]:
def evaluate_anomaly_detector(
    scores: np.ndarray,
    y_true: np.ndarray,
    method_name: str = 'Detector',
    n_thresholds: int = N_THRESHOLDS,
) -> dict:
    '''Evaluate an anomaly detector using PR-curve and ROC-AUC metrics.

    Sweeps over thresholds to find the best F1 score, and computes
    Average Precision (area under the PR curve) and ROC-AUC.

    Args:
        scores:       Anomaly scores of shape (n,). Higher = more anomalous.
        y_true:       Ground-truth labels of shape (n,). 1 = anomaly.
        method_name:  Name for display in output.
        n_thresholds: Number of threshold values to sweep.

    Returns:
        Dict with keys: method, ap, roc_auc, best_f1, best_threshold,
        precision_curve, recall_curve, thresholds_curve.
    '''
    ap      = float(average_precision_score(y_true, scores))
    roc_auc = float(roc_auc_score(y_true, scores))

    # Sweep thresholds for best F1
    thresholds  = np.linspace(scores.min(), scores.max(), n_thresholds)
    best_f1     = 0.0
    best_thresh = float(thresholds[0])
    best_preds  = np.zeros_like(y_true)

    for t in thresholds:
        preds = (scores >= t).astype(int)
        f1    = float(f1_score(y_true, preds, zero_division=0))
        if f1 > best_f1:
            best_f1     = f1
            best_thresh = float(t)
            best_preds  = preds

    # PR curve
    prec, rec, pr_thresh = precision_recall_curve(y_true, scores)

    result = {
        'method':            method_name,
        'ap':                round(ap, 4),
        'roc_auc':           round(roc_auc, 4),
        'best_f1':           round(best_f1, 4),
        'best_threshold':    round(best_thresh, 6),
        'best_preds':        best_preds,
        'precision_curve':   prec,
        'recall_curve':      rec,
        'thresholds_curve':  pr_thresh,
    }
    print(f'{method_name:30s}  AP={ap:.4f}  ROC-AUC={roc_auc:.4f}  F1={best_f1:.4f}')
    return result


# ── Evaluate all methods ──────────────────────────────────────────────────────
print('Evaluation results (test set):')
print('-' * 65)
result_z     = evaluate_anomaly_detector(scores_z,     y_test, 'Z-Score (max feature)')
result_iqr   = evaluate_anomaly_detector(scores_iqr,   y_test, 'IQR Rule')
result_gauss = evaluate_anomaly_detector(scores_gauss, y_test, 'Gaussian MLE')
result_if    = evaluate_anomaly_detector(scores_if,    y_test, 'Isolation Forest (scratch)')

# ── sklearn Isolation Forest for comparison ───────────────────────────────────
skl_if = SklearnIF(n_estimators=N_TREES, max_samples=MAX_SAMPLES_IF,
                   random_state=SEED, contamination=ANOMALY_FRAC)
skl_if.fit(X_train)
# sklearn decision_function: more negative = more anomalous; negate for our convention
skl_scores  = -skl_if.decision_function(X_test)
result_skl  = evaluate_anomaly_detector(skl_scores, y_test, 'Isolation Forest (sklearn)')

all_results = [result_z, result_iqr, result_gauss, result_if, result_skl]

### Sanity Check: Confusion Matrix for Each Method

With the best F1 threshold applied to each method, we inspect the confusion matrices to understand the error breakdown — are mistakes mostly false positives (normal flagged as anomalous) or false negatives (missed anomalies)?

In [ ]:
# ── Confusion matrices at best-F1 threshold ───────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
method_subset = [result_z, result_iqr, result_gauss, result_if]

for ax, res in zip(axes, method_subset):
    cm = confusion_matrix(y_test, res['best_preds'])
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Normal', 'Anomaly'], fontsize=9)
    ax.set_yticklabels(['Normal', 'Anomaly'], fontsize=9)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontsize=14, color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_title(f"{res['method'].replace(' (scratch)', '')}\n"
                 f"F1={res['best_f1']:.3f}", fontsize=10)

plt.suptitle('Confusion Matrices at Best-F1 Threshold', fontsize=13,
             fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 3 — Evaluation on the Test Dataset

We now visualise the methods on the full test set and compare the precision-recall curves side by side.

In [ ]:
# ── Precision-Recall curves ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors_methods = ['royalblue', 'seagreen', 'darkorange', 'tomato', 'purple']
ax = axes[0]
for res, col in zip(all_results, colors_methods):
    ax.plot(
        res['recall_curve'], res['precision_curve'],
        lw=2, color=col,
        label=f"{res['method']} (AP={res['ap']:.3f})",
    )
base_rate = y_test.mean()
ax.axhline(base_rate, color='grey', linestyle='--', lw=1.2,
           label=f'Baseline (AP={base_rate:.3f})')
ax.set_xlabel('Recall',    fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Curves', fontsize=13)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.legend(fontsize=8, loc='upper right')

# ── Score distributions ───────────────────────────────────────────────────────
ax = axes[1]
all_scores = [scores_z, scores_iqr, scores_gauss, scores_if]
names_short = ['Z-Score', 'IQR', 'Gaussian', 'IsoForest']
for i, (s, nm) in enumerate(zip(all_scores, names_short)):
    # Normalise each score to [0, 1] for comparison
    s_norm = (s - s.min()) / (s.max() - s.min() + 1e-10)
    ax.hist(s_norm[y_test == 0], bins=25, alpha=0.35, color=colors_methods[i],
            density=True, label=nm + ' (Normal)')
    ax.hist(s_norm[y_test == 1], bins=12, alpha=0.6, color=colors_methods[i],
            density=True, histtype='step', lw=2, label=nm + ' (Anomaly)')

ax.set_xlabel('Normalised Anomaly Score')
ax.set_ylabel('Density')
ax.set_title('Score Distributions (solid=normal, step=anomaly)', fontsize=11)
ax.legend(fontsize=7)

fig.suptitle('Anomaly Detector Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── ROC and AP summary table ──────────────────────────────────────────────────
df_results = pd.DataFrame([
    {'Method': r['method'], 'AP': r['ap'], 'ROC-AUC': r['roc_auc'],
     'Best F1': r['best_f1']}
    for r in all_results
]).set_index('Method')
print('\nSummary table:')
print(df_results.to_string())

In [ ]:
# ── Decision boundary visualisation (2-D only) ────────────────────────────────
resolution = 150
x1_range = np.linspace(X_test[:, 0].min() - 0.5, X_test[:, 0].max() + 0.5, resolution)
x2_range = np.linspace(X_test[:, 1].min() - 0.5, X_test[:, 1].max() + 0.5, resolution)
xx1, xx2 = np.meshgrid(x1_range, x2_range)
X_grid   = np.column_stack([xx1.ravel(), xx2.ravel()])

# Evaluate each detector on the grid (normalise scores to [0,1])
det_names  = ['Z-Score', 'IQR Rule', 'Gaussian', 'IsoForest']
det_scores_grid = [
    zscore_anomaly_scores(X_train, X_grid),
    iqr_anomaly_scores(X_train, X_grid),
    gauss_det.score_samples(X_grid),
    if_scratch.score_samples(X_grid),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (det_score_g, nm) in zip(axes, zip(det_scores_grid, det_names)):
    s_norm = (det_score_g - det_score_g.min()) / (
        det_score_g.max() - det_score_g.min() + 1e-10
    )
    heat = s_norm.reshape(xx1.shape)
    ax.contourf(xx1, xx2, heat, levels=30, cmap='RdYlGn_r', alpha=0.8)
    ax.scatter(X_test[y_test == 0, 0], X_test[y_test == 0, 1],
               c='white', alpha=0.5, s=12, edgecolors='grey', lw=0.4)
    ax.scatter(X_test[y_test == 1, 0], X_test[y_test == 1, 1],
               c='black', alpha=0.9, s=60, marker='x', lw=2)
    ax.set_title(nm, fontsize=11)
    ax.set_xlabel('F1')
    if ax == axes[0]:
        ax.set_ylabel('F2')

fig.suptitle('Anomaly Score Heatmaps (red=high anomaly, green=normal)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 4 — Evaluation & Analysis

We now go deeper: threshold selection sensitivity, contamination fraction robustness, and an error analysis examining which anomalies each method misses.

In [ ]:
def threshold_sensitivity(
    scores: np.ndarray,
    y_true: np.ndarray,
    n_points: int = 100,
) -> pd.DataFrame:
    '''Evaluate precision, recall and F1 across a sweep of thresholds.

    Args:
        scores:   Anomaly scores of shape (n,).
        y_true:   Binary ground-truth labels of shape (n,).
        n_points: Number of threshold values to evaluate.

    Returns:
        DataFrame with columns threshold, precision, recall, f1.
    '''
    thresholds = np.linspace(scores.min(), scores.max(), n_points)
    rows = []
    for t in thresholds:
        preds = (scores >= t).astype(int)
        tp  = int(np.sum((preds == 1) & (y_true == 1)))
        fp  = int(np.sum((preds == 1) & (y_true == 0)))
        fn  = int(np.sum((preds == 0) & (y_true == 1)))
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
        rows.append({'threshold': t, 'precision': prec, 'recall': rec, 'f1': f1})
    return pd.DataFrame(rows)


# ── Threshold sensitivity for Gaussian and IsoForest ─────────────────────────
df_thresh_g  = threshold_sensitivity(scores_gauss, y_test)
df_thresh_if = threshold_sensitivity(scores_if,    y_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, df_t, nm in zip(axes, [df_thresh_g, df_thresh_if],
                          ['Gaussian MLE', 'Isolation Forest']):
    ax.plot(df_t['threshold'], df_t['precision'], lw=2, color='royalblue', label='Precision')
    ax.plot(df_t['threshold'], df_t['recall'],    lw=2, color='seagreen',  label='Recall')
    ax.plot(df_t['threshold'], df_t['f1'],        lw=2, color='tomato',    label='F1')
    best_idx = df_t['f1'].idxmax()
    best_t   = df_t.loc[best_idx, 'threshold']
    best_f1  = df_t.loc[best_idx, 'f1']
    ax.axvline(best_t, color='black', linestyle='--', lw=1.2,
               label=f'Best F1={best_f1:.3f}')
    ax.set_xlabel('Decision Threshold')
    ax.set_ylabel('Score')
    ax.set_title(f'{nm} — Threshold Sensitivity', fontsize=12)
    ax.set_ylim([0, 1.05])
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Error analysis: which anomalies are missed by each method? ────────────────
thresh_gauss = float(result_gauss['best_threshold'])
thresh_if    = float(result_if['best_threshold'])

preds_gauss = (scores_gauss >= thresh_gauss).astype(int)
preds_if    = (scores_if    >= thresh_if).astype(int)

# Find anomaly test indices
anom_idx = np.where(y_test == 1)[0]

# Categorise anomalies by detection status across methods
both_detected  = anom_idx[(preds_gauss[anom_idx] == 1) & (preds_if[anom_idx] == 1)]
only_gauss     = anom_idx[(preds_gauss[anom_idx] == 1) & (preds_if[anom_idx] == 0)]
only_if        = anom_idx[(preds_gauss[anom_idx] == 0) & (preds_if[anom_idx] == 1)]
both_missed    = anom_idx[(preds_gauss[anom_idx] == 0) & (preds_if[anom_idx] == 0)]

print(f'Anomaly detection breakdown ({len(anom_idx)} total anomalies):')
print(f'  Detected by both        : {len(both_detected)}')
print(f'  Detected by Gaussian only: {len(only_gauss)}')
print(f'  Detected by IsoForest only: {len(only_if)}')
print(f'  Missed by both          : {len(both_missed)}')

# ── Visualise missed anomalies ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
normal_test = X_test[y_test == 0]
ax.scatter(normal_test[:, 0], normal_test[:, 1],
           c=COL_NORMAL, alpha=0.2, s=12, label='Normal')

if len(both_detected) > 0:
    ax.scatter(X_test[both_detected, 0], X_test[both_detected, 1],
               c='seagreen', s=80, marker='D', zorder=5, label='Both detected')
if len(only_gauss) > 0:
    ax.scatter(X_test[only_gauss, 0], X_test[only_gauss, 1],
               c='royalblue', s=80, marker='^', zorder=5, label='Gaussian only')
if len(only_if) > 0:
    ax.scatter(X_test[only_if, 0], X_test[only_if, 1],
               c='darkorange', s=80, marker='s', zorder=5, label='IsoForest only')
if len(both_missed) > 0:
    ax.scatter(X_test[both_missed, 0], X_test[both_missed, 1],
               c='black', s=80, marker='x', lw=2, zorder=5, label='Both missed')

ax.set_title('Anomaly Detection Error Analysis\n(Gaussian vs Isolation Forest)',
             fontsize=12)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# ── Where do the missed anomalies lie relative to the training distribution? ──
if len(both_missed) > 0:
    missed_X = X_test[both_missed]
    print(f'\nMissed anomalies Mahalanobis distances:')
    missed_maha = gauss_det.mahalanobis(missed_X)
    print(f'  Mean: {missed_maha.mean():.3f},  Min: {missed_maha.min():.3f}')
    print('  These anomalies overlap with the normal distribution (hard cases).')
else:
    print('\nAll anomalies detected by at least one method.')

In [ ]:
# ── Ablation: effect of n_trees and max_samples on IF performance ─────────────
print('Ablation: Isolation Forest hyperparameter sensitivity...')

n_trees_list   = [10, 25, 50, 100]
max_samp_list  = [32, 64, 128, 256]
ablation_rows  = []

# Trees ablation (fixed max_samples=256)
for nt in n_trees_list:
    if_abl = IsolationForest(n_estimators=nt, max_samples=256,
                             random_state=SEED)
    if_abl.fit(X_train)
    s_abl  = if_abl.score_samples(X_test)
    ap_abl = float(average_precision_score(y_test, s_abl))
    ablation_rows.append({'n_trees': nt, 'max_samples': 256, 'AP': round(ap_abl, 4)})
    print(f'  n_trees={nt:3d}, max_samples=256: AP={ap_abl:.4f}')

# Max-samples ablation (fixed n_trees=100)
for ms in max_samp_list:
    if_abl = IsolationForest(n_estimators=50, max_samples=ms,
                             random_state=SEED)
    if_abl.fit(X_train)
    s_abl  = if_abl.score_samples(X_test)
    ap_abl = float(average_precision_score(y_test, s_abl))
    ablation_rows.append({'n_trees': 50, 'max_samples': ms, 'AP': round(ap_abl, 4)})
    print(f'  n_trees= 50, max_samples={ms:3d}: AP={ap_abl:.4f}')

df_ablation = pd.DataFrame(ablation_rows)
print('\nAblation summary:')
print(df_ablation.to_string(index=False))

In [ ]:
# ── LOF k-sensitivity study ───────────────────────────────────────────────────
print('LOF k-sensitivity study...')
k_values    = [5, 10, 15, 20, 30, 50]
lof_k_rows  = []

for k in k_values:
    lof_k = LocalOutlierFactor(n_neighbors=k)
    lof_k.fit(X_train[y_train == 0])
    s_k   = lof_k.score_samples(X_test)
    ap_k  = float(average_precision_score(y_test, s_k))
    # Best F1 at this k
    thrs  = np.linspace(s_k.min(), s_k.max(), 100)
    best_f1_k = max(
        float(f1_score(y_test, (s_k >= t).astype(int), zero_division=0))
        for t in thrs
    )
    lof_k_rows.append({'k': k, 'AP': round(ap_k, 4), 'best_F1': round(best_f1_k, 4)})
    print(f'  k={k:3d}: AP={ap_k:.4f}  F1={best_f1_k:.4f}')

df_lof_k = pd.DataFrame(lof_k_rows).set_index('k')
print('\nLOF k-sensitivity:')
print(df_lof_k.to_string())

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(df_lof_k.index, df_lof_k['AP'],      'o-', color='royalblue', lw=2, label='AP')
ax.plot(df_lof_k.index, df_lof_k['best_F1'], 's--', color='tomato',   lw=2, label='Best F1')
ax.set_xlabel('n_neighbors (k)')
ax.set_ylabel('Score')
ax.set_title('LOF Sensitivity to k', fontsize=12)
ax.set_ylim([0, 1.0])
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


# ── High-dimensional robustness (curse of dimensionality) ─────────────────────
print('\nHigh-dimensional robustness study (d = 2, 5, 10, 20, 50)...')
dim_rows = []

for d in [2, 5, 10, 20, 50]:
    X_d, y_d = make_classification(
        n_samples=600,
        n_features=d,
        n_informative=d,
        n_redundant=0,
        n_clusters_per_class=1,
        weights=[0.95, 0.05],
        flip_y=0.0,
        class_sep=1.5,
        random_state=SEED,
    )
    X_d = StandardScaler().fit_transform(X_d)
    X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
        X_d, y_d, test_size=0.3, random_state=SEED, stratify=y_d
    )

    # Gaussian
    gd_d = GaussianAnomalyDetector()
    gd_d.fit(X_tr_d[y_tr_d == 0])
    s_gd  = gd_d.score_samples(X_te_d)
    ap_gd = float(average_precision_score(y_te_d, s_gd)) if y_te_d.sum() > 0 else 0.0

    # Isolation Forest
    if_d = IsolationForest(n_estimators=50,
                           max_samples=min(128, X_tr_d.shape[0]),
                           random_state=SEED)
    if_d.fit(X_tr_d)
    s_ifd  = if_d.score_samples(X_te_d)
    ap_ifd = float(average_precision_score(y_te_d, s_ifd)) if y_te_d.sum() > 0 else 0.0

    # Mahalanobis normalised distance (ratio max/median for normals — dimensionality indicator)
    maha_normal  = gd_d.mahalanobis(X_te_d[y_te_d == 0])
    maha_anomaly = gd_d.mahalanobis(X_te_d[y_te_d == 1])
    sep = (maha_anomaly.mean() - maha_normal.mean()) / (maha_normal.std() + 1e-10)

    dim_rows.append({
        'd':               d,
        'AP_Gaussian':     round(ap_gd,  4),
        'AP_IsoForest':    round(ap_ifd, 4),
        'Maha_separation': round(float(sep), 3),
    })
    print(f'  d={d:3d}: Gaussian AP={ap_gd:.4f}, IF AP={ap_ifd:.4f}, '
          f'Maha-sep={sep:.3f}')

df_dims = pd.DataFrame(dim_rows).set_index('d')
print('\nDimensionality robustness:')
print(df_dims.to_string())

In [ ]:
# ── Contamination fraction robustness study ───────────────────────────────────
# How do methods perform as the true anomaly fraction changes?
print('Contamination fraction robustness (regenerating data at each fraction)...')

contamination_fracs = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20]
contam_rows         = []

for frac in contamination_fracs:
    X_c, y_c = make_classification(
        n_samples=N_SAMPLES,
        n_features=N_FEATURES,
        n_informative=N_FEATURES,
        n_redundant=0,
        n_clusters_per_class=1,
        weights=[1 - frac, frac],
        flip_y=0.0,
        class_sep=1.5,
        random_state=SEED + 1,
    )
    X_c = StandardScaler().fit_transform(X_c)
    X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
        X_c, y_c, test_size=0.3, random_state=SEED, stratify=y_c
    )

    # Gaussian MLE — fit on normals only
    gd_c = GaussianAnomalyDetector()
    gd_c.fit(X_tr_c[y_tr_c == 0])
    s_g  = gd_c.score_samples(X_te_c)
    ap_g = float(average_precision_score(y_te_c, s_g)) if y_te_c.sum() > 0 else 0.0

    # Isolation Forest — unsupervised (no label information used)
    if_c = IsolationForest(n_estimators=50,
                           max_samples=min(256, X_tr_c.shape[0]),
                           random_state=SEED)
    if_c.fit(X_tr_c)
    s_if  = if_c.score_samples(X_te_c)
    ap_if = float(average_precision_score(y_te_c, s_if)) if y_te_c.sum() > 0 else 0.0

    # Z-Score baseline
    s_z  = zscore_anomaly_scores(X_tr_c, X_te_c)
    ap_z = float(average_precision_score(y_te_c, s_z)) if y_te_c.sum() > 0 else 0.0

    # Random baseline
    rng_b  = np.random.RandomState(SEED)
    s_rnd  = rng_b.rand(len(y_te_c))
    ap_rnd = float(average_precision_score(y_te_c, s_rnd)) if y_te_c.sum() > 0 else 0.0

    contam_rows.append({
        'frac':          frac,
        'n_anomalies':   int(y_te_c.sum()),
        'AP_Gaussian':   round(ap_g,   4),
        'AP_IsoForest':  round(ap_if,  4),
        'AP_ZScore':     round(ap_z,   4),
        'AP_Random':     round(ap_rnd, 4),
    })
    print(f'  frac={frac:.2f}: n_anom={y_te_c.sum():3d}  '
          f'Gauss={ap_g:.3f}  IF={ap_if:.3f}  Z={ap_z:.3f}  Rnd={ap_rnd:.3f}')

df_contam = pd.DataFrame(contam_rows)

# ── Plot contamination robustness ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
fracs_pct = df_contam['frac'].values * 100
ax.plot(fracs_pct, df_contam['AP_Gaussian'],  'o-', color='darkorange', lw=2,
        label='Gaussian MLE')
ax.plot(fracs_pct, df_contam['AP_IsoForest'], 's-', color='steelblue',  lw=2,
        label='Isolation Forest')
ax.plot(fracs_pct, df_contam['AP_ZScore'],    '^-', color='seagreen',   lw=2,
        label='Z-Score')
ax.plot(fracs_pct, df_contam['AP_Random'],    'd--', color='grey', lw=1.5,
        label='Random baseline')
ax.set_xlabel('True Anomaly Fraction (%)')
ax.set_ylabel('Average Precision (AP)')
ax.set_title('Method Robustness — AP vs Contamination', fontsize=12)
ax.set_ylim([0, 1.05])
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(fracs_pct, df_contam['n_anomalies'], 'o-', color='tomato', lw=2)
ax.set_xlabel('True Anomaly Fraction (%)')
ax.set_ylabel('Number of Test Anomalies')
ax.set_title('Test Set Anomaly Count vs Contamination', fontsize=12)
ax2 = ax.twinx()
ax2.plot(fracs_pct, fracs_pct, '--', color='slategray', lw=1.5, alpha=0.7)
ax2.set_ylabel('Anomaly % (diagonal reference)')

plt.tight_layout()
plt.show()

print('\nKey insight: as contamination increases, AP of Gaussian deteriorates')
print('faster than IF because the Gaussian was fit on normals only.')
print(df_contam.set_index('frac').to_string())

In [ ]:
# ── Comprehensive final comparison table ──────────────────────────────────────
# All methods evaluated on the original test set
all_method_scores = {
    'Z-Score':            scores_z,
    'IQR Rule':           scores_iqr,
    'Gaussian MLE':       scores_gauss,
    'Isolation Forest':   scores_if,
    'LOF (k=20)':         scores_lof,
    'KDE (Gaussian)':     scores_kde,
    'IF (sklearn)':       skl_scores,
}

final_rows = []
for name, s in all_method_scores.items():
    ap      = float(average_precision_score(y_test, s))
    auc     = float(roc_auc_score(y_test, s))
    thrs    = np.linspace(s.min(), s.max(), 200)
    best_f1 = max(
        float(f1_score(y_test, (s >= t).astype(int), zero_division=0))
        for t in thrs
    )
    sep     = (s[y_test == 1].mean() - s[y_test == 0].mean()) / (
        s[y_test == 0].std() + 1e-10
    )
    final_rows.append({
        'Method':     name,
        'AP':         round(ap,      4),
        'ROC-AUC':    round(auc,     4),
        'Best F1':    round(best_f1, 4),
        'Score sep.': round(float(sep), 3),
    })

df_final = pd.DataFrame(final_rows).set_index('Method').sort_values('AP', ascending=False)
print('Final comprehensive comparison (sorted by AP):')
print(df_final.to_string())

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
methods = df_final.index.tolist()
x       = np.arange(len(methods))
width   = 0.27
ap_vals = df_final['AP'].values
f1_vals = df_final['Best F1'].values
auc_vals = df_final['ROC-AUC'].values

ax.bar(x - width, ap_vals,  width, label='AP',       color='royalblue', alpha=0.85)
ax.bar(x,          f1_vals,  width, label='Best F1',  color='seagreen',  alpha=0.85)
ax.bar(x + width, auc_vals, width, label='ROC-AUC',  color='darkorange', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim([0, 1.1])
ax.set_title('All Detectors — AP, Best F1, and ROC-AUC', fontsize=12)
ax.legend(fontsize=9)
ax.axhline(float(y_test.mean()), color='grey', linestyle='--', lw=1.2, alpha=0.7,
           label=f'Random baseline AP={y_test.mean():.3f}')
plt.tight_layout()
plt.show()

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Use Precision-Recall, not accuracy or ROC-AUC, for anomaly evaluation.** With 5% anomalies, even a trivial "flag nothing" classifier achieves 95% accuracy. AP (area under the PR curve) directly measures performance on the rare class.

2. **Method complexity should match data structure.** Z-score and IQR work well for 1-D features or obvious magnitude outliers. The multivariate Gaussian handles correlated features but assumes a single ellipsoidal normal distribution. Isolation Forest is non-parametric and effective when anomalies differ in complex ways.

3. **Isolation Forest exploits anomaly rarity and structure.** Anomalies are isolated in fewer random splits than dense normal points — no distributional assumption is required. Smaller sub-sample sizes (e.g., 256) often outperform full-sample trees due to reduced masking effects.

4. **Threshold selection is a design decision, not a free parameter.** The right threshold depends on the cost of false positives vs false negatives. Plotting the full precision-recall curve and presenting AP gives a threshold-independent view.

5. **Anomaly detection and density estimation are deeply connected.** The Gaussian detector computes $\log p(\mathbf{x})$ directly. GMMs (03-06) extend this to multi-modal distributions and form the statistical foundation for more sophisticated unsupervised anomaly detectors.

### What's Next

→ **03-08 (Kernel Methods)** revisits density estimation in infinite-dimensional feature spaces via kernel density estimation (KDE), which provides a non-parametric alternative to Gaussian density fitting.

### Going Further

- Liu, Ting & Zhou (2008) *Isolation Forest* — the original paper.
- Chandola, Banerjee & Kumar (2009) *Anomaly Detection: A Survey* — comprehensive overview.
- Pimentel et al. (2014) *A Review of Novelty Detection* — novelty vs anomaly detection.
- Autoencoders for anomaly detection: reconstruction error as an anomaly score (see Module 11-01).